# 01 Extraction
Initial notebook for data extraction steps and profiling.

## Section 1: Imports and Configuration
Import necessary libraries and set pandas display options for data exploration.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

RAW_PATH = '../data/raw/'
FILES = [
    'results.csv',
    'lap_times.csv',
    'pit_stops.csv',
    'qualifying.csv',
    'races.csv',
    'circuits.csv',
    'constructors.csv',
    'drivers.csv',
    'status.csv',
    'constructor_standings.csv',
    'driver_standings.csv',
    'constructor_results.csv',
    'sprint_results.csv',
    'seasons.csv'
]

## Section 2: Load All Files
Load all 14 CSV files with `dtype=str` to preserve `\N` strings and display a summary table of their shapes.

In [2]:
raw = {}
summary_data = []

for file in FILES:
    file_path = os.path.join(RAW_PATH, file)
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, dtype=str)
        name = file.replace('.csv', '')
        raw[name] = df
        
        size_kb = os.path.getsize(file_path) / 1024
        summary_data.append({
            'File Name': file,
            'Rows': len(df),
            'Columns': len(df.columns),
            'Size (KB)': round(size_kb, 2)
        })
    else:
        print(f"Warning: {file} not found at {file_path}")

summary_df = pd.DataFrame(summary_data)
display(summary_df)

,File Name,Rows,Columns,Size (KB)
0,results.csv,27304,18,1615.22
1,lap_times.csv,618766,6,16909.61
2,pit_stops.csv,22193,7,743.71
3,qualifying.csv,11036,9,439.77
4,races.csv,1171,18,169.07
5,circuits.csv,78,9,9.97
6,constructors.csv,214,5,17.23
7,drivers.csv,865,9,92.62
8,status.csv,140,2,2.10
9,constructor_standings.csv,13664,7,289.90


## Section 3: \N Inventory
Calculate the total count and percentage of literal `\N` strings across all columns in each dataframe.

In [3]:
n_inventory_data = []
total_n_count = 0

for name, df in raw.items():
    n_counts = (df == r'\N').sum()
    cols_with_n = n_counts[n_counts > 0]
    
    for col, count in cols_with_n.items():
        total_rows = len(df)
        percentage = (count / total_rows) * 100
        n_inventory_data.append({
            'File Name': f"{name}.csv",
            'Column Name': col,
            '\\N Count': count,
            '% of Column': round(percentage, 2)
        })
        total_n_count += count

n_inventory_df = pd.DataFrame(n_inventory_data)
display(n_inventory_df)
print(f"Total \\N count across all files: {total_n_count}")

,File Name,Column Name,\N Count,% of Column
0,results.csv,number,6,0.02
1,results.csv,grid,20,0.07
2,results.csv,position,10953,40.12
3,results.csv,time,19252,70.51
4,results.csv,milliseconds,19252,70.51
5,results.csv,fastestLap,18535,67.88
6,results.csv,rank,18277,66.94
7,results.csv,fastestLapTime,18535,67.88
8,results.csv,fastestLapSpeed,19052,69.78
9,pit_stops.csv,duration,3,0.01


Total \N count across all files: 162385


**Understanding `\N` in the Ergast Database**

In the Ergast F1 database, a literal string `\N` is used to represent missing or null values. This is an explicit convention of the dataset's SQL export format. It is critically important to understand that pandas does not automatically recognize `\N` as a null value (`NaN` or `pd.NA`). If these are not explicitly handled and replaced before converting columns to numeric data types, pandas will either throw an error or coerce them incorrectly, compromising all downstream calculations. They must be replaced with `pd.NA` or `np.nan` during the cleaning phase.

## Section 4: Shape and Dtype Audit
Audit the data types, unique values, and true pandas null counts for each column, highlighting potential numeric columns currently stored as objects.

In [4]:
numeric_indicators = ['grid', 'position', 'points', 'milliseconds', 'id', 'time', 'lap', 'stop', 'year', 'round', 'rank', 'speed']

for name, df in raw.items():
    print(f"--- Audit for {name}.csv ---")
    audit_data = []
    for col in df.columns:
        dtype = df[col].dtype
        unique_count = df[col].nunique()
        sample_vals = df[col].dropna().unique()[:3]
        null_count = df[col].isnull().sum()
        
        flag = ""
        if dtype == 'object' and any(ind in col.lower() for ind in numeric_indicators):
            flag = "FLAG: Likely Numeric"
            
        audit_data.append({
            'Column Name': col,
            'Dtype': dtype,
            'Unique Values': unique_count,
            'Sample (3)': list(sample_vals),
            'True Null Count': null_count,
            'Notes': flag
        })
    audit_df = pd.DataFrame(audit_data)
    display(audit_df)
    print("\n")

--- Audit for results.csv ---


,Column Name,Dtype,Unique Values,Sample (3),True Null Count,Notes
0,resultId,str,27304,"[1, 2, 3]",0,
1,raceId,str,1152,"[18, 19, 20]",0,
2,driverId,str,865,"[1, 2, 3]",0,
3,constructorId,str,213,"[1, 2, 3]",0,
4,number,str,130,"[22, 3, 7]",0,
5,grid,str,36,"[1, 5, 7]",0,
6,position,str,34,"[1, 2, 3]",0,
7,positionText,str,39,"[1, 2, 3]",0,
8,positionOrder,str,39,"[1, 2, 3]",0,
9,points,str,39,"[10, 8, 6]",0,




--- Audit for lap_times.csv ---


,Column Name,Dtype,Unique Values,Sample (3),True Null Count,Notes
0,raceId,str,571,"[841, 842, 843]",0,
1,driverId,str,147,"[20, 1, 17]",0,
2,lap,str,87,"[1, 2, 3]",0,
3,position,str,24,"[1, 3, 4]",0,
4,time,str,76617,"[1:38.109, 1:33.006, 1:32.713]",0,
5,milliseconds,str,76617,"[98109, 93006, 92713]",0,




--- Audit for pit_stops.csv ---


...

,Column Name,Dtype,Unique Values,Sample (3),True Null Count,Notes
0,constructorResultsId,str,12898,"[1, 2, 3]",0,
1,raceId,str,1087,"[18, 19, 20]",0,
2,constructorId,str,177,"[1, 2, 3]",0,
3,points,str,61,"[14, 8, 9]",0,
4,status,str,2,"[\N, D]",0,




--- Audit for sprint_results.csv ---


,Column Name,Dtype,Unique Values,Sample (3),True Null Count,Notes
0,resultId,str,502,"[1, 2, 3]",0,
1,raceId,str,25,"[1061, 1065, 1071]",0,
2,driverId,str,36,"[830, 1, 822]",0,
3,constructorId,str,14,"[9, 131, 6]",0,
4,number,str,36,"[33, 44, 77]",0,
5,grid,str,23,"[2, 1, 3]",0,
6,position,str,23,"[1, 2, 3]",0,
7,positionText,str,23,"[1, 2, 3]",0,
8,positionOrder,str,22,"[1, 2, 3]",0,
9,points,str,9,"[3, 2, 1]",0,




--- Audit for seasons.csv ---


,Column Name,Dtype,Unique Values,Sample (3),True Null Count,Notes
0,year,str,77,"[2009, 2008, 2007]",0,
1,url,str,77,[http://en.wikipedia.org/wiki/2009_Formula_One...,0,


## Section 5: Year Range Verification
Merge key tables with the races table to determine the temporal coverage of results, pit stops, lap times, and qualifying data.

In [5]:
if 'races' in raw:
    races_df = raw['races']
    
    if 'results' in raw:
        res_races = pd.merge(raw['results'], races_df, on='raceId', how='inner')
        min_yr = res_races['year'].astype(int).min()
        max_yr = res_races['year'].astype(int).max()
        print(f"Results data year range: {min_yr} to {max_yr}")
        
    if 'pit_stops' in raw:
        pit_races = pd.merge(raw['pit_stops'], races_df, on='raceId', how='inner')
        min_yr = pit_races['year'].astype(int).min()
        max_yr = pit_races['year'].astype(int).max()
        print(f"Pit Stops data year range: {min_yr} to {max_yr}")
        
    if 'lap_times' in raw:
        lap_races = pd.merge(raw['lap_times'], races_df, on='raceId', how='inner')
        min_yr = lap_races['year'].astype(int).min()
        max_yr = lap_races['year'].astype(int).max()
        print(f"Lap Times data year range: {min_yr} to {max_yr}")
        
    if 'qualifying' in raw:
        qual_races = pd.merge(raw['qualifying'], races_df, on='raceId', how='inner')
        min_yr = qual_races['year'].astype(int).min()
        max_yr = qual_races['year'].astype(int).max()
        print(f"Qualifying data year range: {min_yr} to {max_yr}")

Results data year range: 1950 to 2026
Pit Stops data year range: 1994 to 2026
Lap Times data year range: 1996 to 2026
Qualifying data year range: 1994 to 2026


## Section 6: Key Relationship Checks
Verify referential integrity by checking if foreign keys in the results table exist in their respective dimension tables.

In [6]:
if 'results' in raw:
    results_df = raw['results']
    
    if 'races' in raw:
        orphan_races = ~results_df['raceId'].isin(raw['races']['raceId'])
        orphan_count = orphan_races.sum()
        if orphan_count == 0:
            print("Check: every raceId in results exists in races -> PASS")
        else:
            print(f"Check: every raceId in results exists in races -> FAIL: {orphan_count} orphan records")
            
    if 'constructors' in raw:
        orphan_constructors = ~results_df['constructorId'].isin(raw['constructors']['constructorId'])
        orphan_count = orphan_constructors.sum()
        if orphan_count == 0:
            print("Check: every constructorId in results exists in constructors -> PASS")
        else:
            print(f"Check: every constructorId in results exists in constructors -> FAIL: {orphan_count} orphan records")
            
    if 'races' in raw and 'circuits' in raw:
        orphan_circuits = ~raw['races']['circuitId'].isin(raw['circuits']['circuitId'])
        orphan_count = orphan_circuits.sum()
        if orphan_count == 0:
            print("Check: every circuitId in races exists in circuits -> PASS")
        else:
            print(f"Check: every circuitId in races exists in circuits -> FAIL: {orphan_count} orphan records")
            
    if 'status' in raw:
        orphan_status = ~results_df['statusId'].isin(raw['status']['statusId'])
        orphan_count = orphan_status.sum()
        if orphan_count == 0:
            print("Check: every statusId in results exists in status -> PASS")
        else:
            print(f"Check: every statusId in results exists in status -> FAIL: {orphan_count} orphan records")

Check: every raceId in results exists in races -> PASS
Check: every constructorId in results exists in constructors -> PASS
Check: every circuitId in races exists in circuits -> PASS
Check: every statusId in results exists in status -> PASS


## Section 7: Columns of Interest Preview
Preview the distribution of specific key columns in the results dataset to understand grid positions, final classifications, and status codes.

In [7]:
if 'results' in raw:
    cols_to_preview = ['grid', 'positionText', 'statusId']
    for col in cols_to_preview:
        if col in raw['results'].columns:
            print(f"--- Top 10 Value Counts for {col} ---")
            display(raw['results'][col].value_counts().head(10))
            print("\n")

--- Top 10 Value Counts for grid ---


grid
0     1638
1     1162
7     1161
5     1158
11    1158
4     1158
9     1158
3     1156
10    1156
8     1155
Name: count, dtype: int64



--- Top 10 Value Counts for positionText ---


positionText
R    8952
F    1368
3    1162
4    1162
2    1160
5    1158
1    1155
6    1151
7    1131
8    1103
Name: count, dtype: int64



--- Top 10 Value Counts for statusId ---


statusId
1     8035
11    4131
5     2033
12    1626
3     1076
81    1025
4      865
6      814
20     795
13     733
Name: count, dtype: int64

**Why `positionText` is Richer than `position`**

The `positionText` column provides a more nuanced view of a driver's classification than the numeric `position` column. While `position` only contains integers for drivers who finished or were classified, `positionText` includes codes like 'R' (Retired), 'D' (Disqualified), 'E' (Excluded), 'W' (Withdrawn), 'F' (Failed to qualify), and 'N' (Not classified). This makes it essential for accurately distinguishing between different types of non-finishers and understanding the exact nature of a DNF (Did Not Finish), which the numeric `position` column obscures by simply recording a `\N`.

## Section 8: Summary

**Total rows across all files:** 726,600+

**Total `\N` values found:** Computed in execution, but expected to be high.

**Year range of the dataset:** 1950 to present (pit stops available 1994 onwards).

**Key Data Issues Identified:**
1. `\N` is a LITERAL STRING (Ergast convention) — not a real null. Requires explicit handling before any numeric operations.
2. `results.grid = 0` means pit lane start (1,638 rows) — exclude from delta calculations but keep in dataset.
3. `results.position` has `\N` for non-finishers INCLUDING lapped cars. Use `positionOrder` (always int, range 1–39, zero nulls) for ranking.
4. True DNF = `statusId` NOT IN `{1,11,12,13,14,15,16,17,18,19}`. Status 1=Finished, 11–19=N Laps down (still classified finishers).
5. `qualifying` `q1`/`q2`/`q3` are time strings like "1:26.572" — must be parsed to milliseconds.
6. `pit_stops` data starts 1994 — for density, scope pit KPIs to 2010+.
7. `constructor_results.status` column is 99.9% `\N` — do not use it. For constructor DNFs, join results → status via `statusId` instead.
8. Qualifying `Q2`/`Q3` are `\N` for drivers eliminated early — expected, not a data error.

**Dataset Status:**
`circuits.csv`, `constructors.csv`, and `status.csv` are mostly clean. The remaining tables (especially `results.csv`, `lap_times.csv`, `pit_stops.csv`, and `qualifying.csv`) require significant transformation in the cleaning phase.